# Phase 6 — Landmark Preference Subpopulation (DREADD saline/DCZ cohort)

Track B only, same reasoning as Phases 4/5 — Track A's version is deferred until Track A gets revisited across Phases 3–6 after this phase.

RSC is classically implicated in landmark/boundary-vector spatial coding. This phase set out to ask whether DCZ's effect on V1 population SMI (established in Phase 4, decomposed by anatomical layer in Phase 5) is concentrated in the subpopulation of V1 cells whose spatial firing is landmark-anchored. It ended up somewhere more specific: a well-characterized, statistically robust, but causally unresolved and non-replicating pattern in landmark-4 (reward-proximal) preference specifically — see Findings below.

Reuses `identify_landmark_responses` from `3.LandmarkPreference/LandmarkPrefernce_SingleSessionAnalysis.py` **unchanged, via import** — that file's name doesn't start with a digit, so it's importable, unlike this project's own phase-numbered files.

Landmark config for this cohort's VR track: `landmark_positions=[37, 65, 93, 120]` cm, windows `{before:25, after:0}` for all four — corrected from an initial `[25,55,85,115]`/`{before:15-20,after:10}` config after Function 6.0's diagnostic plot showed the original landmark-1 window was picking up onset-adjacent activity rather than a genuine landmark response (matching the same swap made directly in `3.LandmarkPreference/LandmarkPrefernce_SingleSessionAnalysis.py`'s own `__main__` block).

**Important**: landmark identification needs the RAW (cm-scale) `bin_centers` straight from `preproc.h5` — NOT Phase 3's saved `*_smi_results_dreadd.h5`, whose stored `bin_centers` is internally rescaled for its own curve fitting.

Structure (built incrementally; consolidate into `6.LandmarkPreference.py` once validated)
------------------------------------------------------------------------------------------
- **Setup** — `discover_smi_sessions`, `load_all_group_dfs_from_phase4` (both reimplemented unchanged from Phase 4/5).
- **6.0** `identify_session_landmark_preference` — per-session wrapper around `identify_landmark_responses`.
- **6.1** `build_landmark_lookup_for_animal` — loops 6.0 over every session for one animal.
- **6.2** `merge_landmark_with_smi_table` — attaches landmark columns onto Phase 4's saved per-cell comparison tables, by `session_label` + `cell_idx`.
- **Composition diagnostics** — `summarize_landmark_composition_by_layer` / `plot_landmark_composition_by_layer` (per layer × condition landmark distribution + chi-square), `run_landmark_composition_analysis_all_groups` (driver across all 5 comparison groups).
- **Population-level figures** — `compute_population_landmark_fraction`, `plot_landmark_last_slope_population`, `plot_landmark_last_slope_by_layer` — the last-landmark-preference reduction under DCZ, population-level and per-layer.
- **Open-loop vs. closed-loop diagnostic** — `summarize_openloop_vs_closedloop_effect` — effect size + per-layer direction reversal, closed-loop (DCZ1/2/3) vs. open-loop (Active_OL/Stationary_OL) groups.
- **Option B** — `build_all_sessions_landmark_smi_table`, `compare_smi_landmark1_vs_last`, `run_landmark1_vs_last_smi_analysis`, `plot_smi_landmark1_vs_last` — within-session (trial-count-free) test of whether landmark-4-preferring cells actually show higher SMI than landmark-1-preferring cells.
- **Trial-count regression** — `get_session_trial_count`, `test_condition_effect_on_landmark_metric`, `test_all_landmark_metrics` — `metric ~ n_trials + C(session_type)`, formally testing whether condition explains anything beyond recording length.
- **Paired significance tests** — `test_paired_significance_population`, `test_paired_significance_by_layer` — paired t-tests (population + per-layer) confirming the saline-vs-dcz pattern is statistically reproducible, not sampling noise.
- **6.Z** save-outputs (standing rule) — `save_dataframe_csv`/`save_figure_png`/`save_json`, `save_all_phase6_outputs`, saved under `Phase6_LandmarkPreference_Results/`.

A GEE (cell-level, clustered-by-session) approach was tried and removed — it sharpened the `has_landmark_preference` DCZ-vs-saline contrast to p=0.082 (JSY093, still not significant) but added complexity without changing the overall picture; the paired t-test approach above is simpler and was kept instead. Per-pair chi-square tests (treating cells as independent within one session) were also tried and discarded — they're pseudo-replicated (hundreds of cells from one session are not independent replicates of that condition) and produced misleadingly small p-values that don't survive proper clustering.

Findings (JSY093, primary; JSY090 as the generalization check) — NOT conclusive
------------------------------------------------------------------------------------
1. **The pattern is real and reproducible in JSY093.** Landmark-4 (reward-proximal, 120cm) preference is lower under DCZ than under its paired saline session in every one of 5 pairs, at the population level and in every individual layer. A paired t-test (the appropriately powered test here, since n=5 *pairs* is the correct independent unit) confirms this isn't sampling noise: population-level p=0.000855; per-layer p=0.0017–0.0200, all four layers significant, L5 strongest/most consistent, L6 consistently weakest.
2. **But "reproducible" does not mean "caused by DCZ."** Saline sessions are the shortest recording of every pair (22–27 trials vs. 55–76 for DCZ, zero overlap), so trial count and condition are perfectly confounded *within* every pair — pairing controls for what differs *between* pairs, not this. The WLS regression built specifically to separate the two (`metric ~ n_trials + C(session_type)`) puts the direct DCZ-vs-saline contrast at p=0.279 for landmark-4 composition (`frac_L4`) — not significant once trial count is in the model. The overall landmark-preferring rate (`fraction_preferring`) fares a little better — DCZ is significantly lower than a drug-free baseline even adjusting for trial count (p=0.002) — but the direct DCZ-vs-saline contrast for that metric isn't significant either (p=0.220 by WLS; p=0.082 by a since-removed GEE model using full cell-level data, the closest anything in this phase came to significance on that specific contrast).
3. **Layer pattern is general, not selective.** All four layers show the reduction, not a subset concentrated in RSC's direct deep-layer targets (L5/L6) — if this is a real DCZ effect, it isn't anatomically selective in the way the original hypothesis predicted. L6 is reliably the weakest layer across several independent diagnostics in this phase (DCZ1's non-significant composition chi-square, Active_OL's near-zero drop, the weakest paired-t p-value here too) — a specific, recurring detail worth remembering, not noise.
4. **Does not replicate in JSY090 — the biggest caveat.** The same closed-loop comparison (DCZ1/2/3) reverses direction in JSY090: 3–4 of 4 layers flip (dcz > saline instead of saline > dcz), essentially a mirror image of JSY093's pattern. Open-loop groups (Active_OL, Stationary_OL) keep the "expected" direction weakly in both animals. A strong, reproducible effect in one animal that inverts in the only other animal tested is evidence against a shared, generalizable DCZ/RSC mechanism as currently characterized — not simply an underpowered non-replication.
5. **The original motivating premise — landmark-4 preference indexes "more spatial" coding — is not supported, and mildly contradicted.** Option B (within-session, trial-count-free by construction) compared SMI between landmark-1- and landmark-4-preferring cells directly: landmark-1 cells show equal or *higher* SMI, significant in JSY090 (p=0.012, 8/9 sessions), directionally consistent in JSY093 (5/7 sessions, p=0.219). A working alternative interpretation (not yet independently tested): landmark 1 and landmark 4 may encode different task-relevant information rather than differing in coding quality — landmark 1 as an early self-localization anchor, landmark 4 as a reward-proximity cue whose salience might build with experience. The clearest way to test that specifically — within-session trial-block dynamics (`analyze_within_session_dynamics` in `3.LandmarkPreference`, unused so far) — was proposed and deferred, not run.
6. **Bottom line**: this phase produced a real, well-characterized, statistically robust phenomenon in JSY093 — not an established DCZ/RSC finding. Its cause (drug vs. the trial-count confound baked into this dataset's design) remains genuinely undetermined, and it does not generalize to the one other animal tested. Resolving this needs either trial-count-matched recordings, more animals, or Track A's paired within-cell design (which sidesteps the population-level trial-count confound entirely, the way it does for Phases 3–5).

Indexing / reuse convention (shared with Phases 0-5)
------------------------------------------------------
Every per-cell array here is indexed by POSITION in `stat[iscell[:, 0] == 1]`, exactly as in every earlier phase. No index translation is ever needed between this phase's inputs (Phase 4's saved `{group}_comparison_table.csv` + each session's raw `preproc.h5`) and its outputs.

In [125]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation\3.LandmarkPreference")

import os
import re
import glob
import json
from collections import Counter
from itertools import combinations

import numpy as np
import h5py
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')  # interactive popups need a real GUI backend
import matplotlib.pyplot as plt
from matplotlib import rcParams
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import kruskal, mannwhitneyu, rankdata, chi2_contingency, wilcoxon, ttest_rel

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Existing pipeline code, reused via import -- never modified.
from helper import files
from LandmarkPrefernce_SingleSessionAnalysis import identify_landmark_responses, plot_cells_by_landmark_assignment

# Landmark config for this cohort's VR track. Updated 2026-08-10 after the
# Function 6.0 diagnostic (plot_cells_by_landmark_assignment) showed the
# original [25,55,85,115]/{before:15-20,after:10} config picked up too much
# onset-adjacent activity in landmark 1's window -- matches the same swap
# made in 3.LandmarkPreference/LandmarkPrefernce_SingleSessionAnalysis.py's
# __main__ block.
LANDMARK_POSITIONS = [37, 65, 93, 120]
LANDMARK_WINDOWS_CONFIG = [
    {'before': 25, 'after': 0},  # landmark 1 at 37cm
    {'before': 25, 'after': 0},  # landmark 2 at 65cm
    {'before': 25, 'after': 0},  # landmark 3 at 93cm
    {'before': 25, 'after': 0},  # landmark 4 at 120cm
]

# One real DREADD animal's already-saved Phase 4 output, to develop and
# sanity-check against.
TEST_ANIMAL_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"

## Setup — `discover_smi_sessions`

Reimplemented unchanged from Phase 4's Function 4.2. Phase 4's saved `{group}_comparison_table.csv` files carry `session_label`/`condition`/per-cell SMI data but not each session's raw TSeries folder path — Function 6.0 needs that path to load the session's own `preproc.h5` for landmark identification, so this discovery step is needed again here, same as Phase 4/5's other re-derivations of small shared utilities (digit-prefixed phase files can't be imported).

- **Input:** `animal_dir`.
- **Output:** `catalog` (`{label: {'save_path', 'session_type', 'tseries_dir'}}`).

In [126]:
def discover_smi_sessions(animal_dir):
    """
    Scan animal_dir for every already-computed *_smi_results_dreadd.h5
    file, labeling each by whichever known naming pattern its TSeries
    folder matches. Reimplemented unchanged from Phase 4's Function 4.2 --
    see markdown above for why this is needed again here.

    Parameters
    ----------
    animal_dir : str

    Returns
    -------
    catalog : dict
        {label: {'save_path': str, 'session_type': str, 'tseries_dir': str}}
    """
    save_paths = sorted(glob.glob(os.path.join(animal_dir, '**', '*_smi_results_dreadd.h5'),
                                   recursive=True))

    entries = []  # (base_label, session_type, save_path, tseries_dir, tseries_name)
    unmatched = []

    for save_path in save_paths:
        tseries_dir = os.path.dirname(save_path)
        tseries_name = os.path.basename(tseries_dir)
        parent_dir = os.path.dirname(tseries_dir)
        parent_name = os.path.basename(parent_dir)

        upper_tseries = tseries_name.upper()

        if 'SAL' in upper_tseries:
            session_type = 'saline'
            base_label = f'{parent_name}_SALINE'
        elif 'DCZ' in upper_tseries:
            session_type = 'dcz'
            base_label = f'{parent_name}_DCZ'
        else:
            day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
            if day_match:
                session_type = 'baseline'
                base_label = f'Day{day_match.group(1)}'
            else:
                session_type = 'unknown'
                base_label = tseries_name
                unmatched.append(base_label)

        entries.append((base_label, session_type, save_path, tseries_dir, tseries_name))

    label_counts = Counter(e[0] for e in entries)

    catalog = {}
    for base_label, session_type, save_path, tseries_dir, tseries_name in entries:
        label = f'{base_label}__{tseries_name}' if label_counts[base_label] > 1 else base_label

        if label in catalog:
            print(f"WARNING: label '{label}' still collides after disambiguation -- "
                  f"keeping {catalog[label]['save_path']}, skipping {save_path}")
            continue

        catalog[label] = {
            'save_path': save_path,
            'session_type': session_type,
            'tseries_dir': tseries_dir,
        }

    print(f"Discovered {len(catalog)} sessions with saved SMI results under {animal_dir}:")
    for label, info in catalog.items():
        print(f"  [{info['session_type']:>8}] {label}  <-  {info['save_path']}")

    collided_labels = [l for l, c in label_counts.items() if c > 1]
    if collided_labels:
        print(f"\n{len(collided_labels)} label(s) had multiple TSeries and were disambiguated: "
              f"{collided_labels}")

    if unmatched:
        print(f"\n{len(unmatched)} session(s) didn't match a known naming pattern "
              f"(labeled 'unknown'): {unmatched}.")

    return catalog

## Function 6.0 — `identify_session_landmark_preference`

Per-session wrapper around `identify_landmark_responses` (imported unchanged from `3.LandmarkPreference`). Loads this session's own **raw** `preproc.h5` directly (`norm_spatial_activity`, `bin_centers` in cm) rather than anything from Phase 3's saved SMI-results h5.

`identify_landmark_responses` already runs over all `n_cells` (same `stat[iscell[:,0]==1]`-position indexing as every other phase) and returns, per cell: `valid_cells` (bool — global peak fell inside some landmark's window, not onset/reward/zero-activity), `preferred_landmark` (int index into `landmark_positions`, -1 if invalid), `preference_strength`. This function just repackages those into a tidy per-cell DataFrame, keeping the raw dict too in case a later diagnostic needs `landmark_responses`/`rejected_cells`/etc.

- **Input:** `tseries_dir` (a session's TSeries folder), landmark config kwargs (defaulting to the confirmed `LANDMARK_POSITIONS`/`LANDMARK_WINDOWS_CONFIG`).
- **Output:** `landmark_df` (`cell_idx`, `has_landmark_preference`, `preferred_landmark_position`, `preference_strength`), `raw_results` (full `identify_landmark_responses` return dict).

In [127]:
def identify_session_landmark_preference(tseries_dir,
                                          landmark_positions=LANDMARK_POSITIONS,
                                          landmark_windows_config=LANDMARK_WINDOWS_CONFIG,
                                          landmark_window=10.0,
                                          boundary_exclusion=(5, 5),
                                          exclude_first_bins=5, exclude_last_bins=5,
                                          smoothing_sigma=1.0):
    """
    Per-session landmark-preference identification. See markdown above.

    Parameters
    ----------
    tseries_dir : str
        A session's TSeries folder (contains *preproc*.h5).
    landmark_positions : list of float
    landmark_windows_config : list of dict
    landmark_window : float
        Fallback symmetric window, unused since landmark_windows_config is provided.
    boundary_exclusion : tuple of float
    exclude_first_bins, exclude_last_bins : int
    smoothing_sigma : float

    Returns
    -------
    landmark_df : pandas.DataFrame
        One row per cell (cell_idx = position in stat[iscell[:,0]==1]):
        has_landmark_preference, preferred_landmark_position, preference_strength.
    raw_results : dict
        Full identify_landmark_responses() return dict.
    """
    preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {tseries_dir}")
    preproc_data = files.read_h5(preproc_files[0])

    normalized_spatial_activity = preproc_data['norm_spatial_activity']
    bin_centers = preproc_data['bin_centers']  # RAW cm scale -- see markdown above

    raw_results = identify_landmark_responses(
        normalized_spatial_activity, bin_centers, landmark_positions,
        landmark_windows_config=landmark_windows_config,
        landmark_window=landmark_window,
        boundary_exclusion=boundary_exclusion,
        smoothing_sigma=smoothing_sigma,
        exclude_first_bins=exclude_first_bins,
        exclude_last_bins=exclude_last_bins,
    )

    n_cells = len(raw_results['valid_cells'])
    has_pref = raw_results['valid_cells']
    preferred_idx = raw_results['preferred_landmark']

    preferred_position = np.full(n_cells, np.nan)
    preferred_position[has_pref] = np.array(landmark_positions)[preferred_idx[has_pref]]

    landmark_df = pd.DataFrame({
        'cell_idx': np.arange(n_cells),
        'has_landmark_preference': has_pref,
        'preferred_landmark_position': preferred_position,
        'preference_strength': raw_results['preference_strength'],
    })

    return landmark_df, raw_results

In [128]:
# --- Quick test on one real session ---
session_catalog = discover_smi_sessions(TEST_ANIMAL_DIR)

test_label = next(iter(session_catalog))
test_tseries_dir = session_catalog[test_label]['tseries_dir']
print(f"\nTesting on: {test_label}  ({test_tseries_dir})")

landmark_df, raw_results = identify_session_landmark_preference(test_tseries_dir)

print("\nhas_landmark_preference counts:")
print(landmark_df['has_landmark_preference'].value_counts())
landmark_df.head(10)

Discovered 15 sessions with saved SMI results under D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD:
  [baseline] Day1__TSeries-07192026-0941-001  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260719_JSY_JSY090_LongitudinalImaging_DREADD_Day1\TSeries-07192026-0941-001\Day1_smi_results_dreadd.h5
  [baseline] Day2  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260720_JSY_JSY090_LongitudinalImaging_DREADD_Day2\TSeries-07202026-1009-001\Day2_smi_results_dreadd.h5
  [baseline] Day3  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260721_JSY_JSY090_LongitudinalImaging_DREADD_Day3\TSeries-07212026-0907-001\Day3_smi_results_dreadd.h5
  [baseline] Day4  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260722_JSY_JSY090_LongitudinalImaging_DREADD_Day4\TSeries-07222026-1831-002\Day4__TSeries-07222026-1831-002_smi_results_dreadd.h5
  [baseline] Day5  <-  D:\V1_SpatialModulation\2p\V1_prism_DREAD

,cell_idx,has_landmark_preference,preferred_landmark_position,preference_strength
0,0,True,65.0,0.045538
1,1,True,37.0,0.009846
2,2,False,NaN,0.000000
3,3,False,NaN,0.000000
4,4,False,NaN,0.000000
5,5,False,NaN,0.000000
6,6,True,120.0,0.017539
7,7,False,NaN,0.000000
8,8,True,37.0,0.021627
9,9,False,NaN,0.000000


## Diagnostic (not a Phase 6 function) — actual per-cell responses by landmark assignment

Before trusting `has_landmark_preference`/`preferred_landmark_position` enough to build on top of them, visualize what's actually driving the Landmark-1-heavy distribution above. Reuses `plot_cells_by_landmark_assignment` **unchanged** from `3.LandmarkPreference` — its own docstring describes exactly this purpose: "helps verify that landmark assignment is working correctly -- cells assigned to L1 should have peaks near L1."

One panel per landmark: every cell assigned to that landmark, normalized (0-1) mean response profile, sorted by peak position, as a heatmap. Green = this landmark's window + position; red dashed = all four landmark positions. If Landmark 1's panel shows peaks piling up right at the left edge of its window (near the onset-exclusion cutoff) rather than centered on 25cm, that supports the onset-adjacent-activity concern rather than a true landmark response.

In [129]:
from LandmarkPrefernce_SingleSessionAnalysis import plot_cells_by_landmark_assignment

# Need bin_centers for the plot -- identify_session_landmark_preference() doesn't
# return it, so reload it directly from the same test session's preproc.h5.
preproc_files = glob.glob(os.path.join(test_tseries_dir, "*preproc*.h5"))
preproc_data = files.read_h5(preproc_files[0])
bin_centers = preproc_data['bin_centers']

fig = plot_cells_by_landmark_assignment(
    raw_results, bin_centers,
    landmark_positions=LANDMARK_POSITIONS,
    trim_start_bins=5, trim_end_bins=5,
)
# plt.show()

## Function 6.1 — `build_landmark_lookup_for_animal`

Loops Function 6.0 over every session needed, keyed by `session_label` so it lines up directly with Phase 4's saved comparison tables (Function 6.2 merges on this key next). Caches by `session_label` rather than recomputing per comparison group -- the same session (e.g. baseline Day5, or a saline/DCZ pair) can appear in multiple named comparison groups (DCZ1/DCZ2/DCZ3/Active_OL/Stationary_OL all reuse baseline), so this avoids rerunning the landmark identification on the same session more than once per animal.

- **Input:** `session_catalog` (from `discover_smi_sessions`), `session_labels` (defaults to every label in the catalog), landmark config kwargs (default to `LANDMARK_POSITIONS`/`LANDMARK_WINDOWS_CONFIG`).
- **Output:** `landmark_lookup` (`{session_label: landmark_df}`).

In [130]:
def build_landmark_lookup_for_animal(session_catalog, session_labels=None,
                                     landmark_positions=LANDMARK_POSITIONS,
                                     landmark_windows_config=LANDMARK_WINDOWS_CONFIG,
                                     landmark_window=10.0,
                                     boundary_exclusion=(5, 5),
                                     exclude_first_bins=5, exclude_last_bins=5,
                                     smoothing_sigma=1.0):
    """
    Loop identify_session_landmark_preference over every session needed,
    keyed by session_label. See markdown above.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    session_labels : list of str, optional
        Defaults to every label in session_catalog.
    landmark_positions, landmark_windows_config, landmark_window,
    boundary_exclusion, exclude_first_bins, exclude_last_bins, smoothing_sigma :
        Passed through to identify_session_landmark_preference.

    Returns
    -------
    landmark_lookup : dict
        {session_label: landmark_df}.
    """
    if session_labels is None:
        session_labels = list(session_catalog.keys())

    landmark_lookup = {}
    for label in session_labels:
        info = session_catalog[label]
        print(f"\n--- {label} ---")
        landmark_df, _ = identify_session_landmark_preference(
            info['tseries_dir'],
            landmark_positions=landmark_positions,
            landmark_windows_config=landmark_windows_config,
            landmark_window=landmark_window,
            boundary_exclusion=boundary_exclusion,
            exclude_first_bins=exclude_first_bins,
            exclude_last_bins=exclude_last_bins,
            smoothing_sigma=smoothing_sigma,
        )
        landmark_lookup[label] = landmark_df
        n_pref = landmark_df['has_landmark_preference'].sum()
        print(f"  {label}: {n_pref}/{len(landmark_df)} cells with a landmark preference")

    print(f"\nBuilt landmark lookup for {len(landmark_lookup)} session(s).")
    return landmark_lookup

In [131]:
# --- Build the landmark lookup for every session already discovered above ---
landmark_lookup = build_landmark_lookup_for_animal(session_catalog)

summary = pd.DataFrame({
    label: {'n_cells': len(df), 'n_preferring': int(df['has_landmark_preference'].sum())}
    for label, df in landmark_lookup.items()
}).T
summary['fraction_preferring'] = summary['n_preferring'] / summary['n_cells']
summary


--- Day1__TSeries-07192026-0941-001 ---

=== LANDMARK PREFERENCE IDENTIFICATION ===
Corridor: 0.5 to 129.5 cm (130 bins)
Bin spacing: 1.00 cm
Global peak exclusion:
  - First 5 bins (< 5.5 cm): onset responses
  - Last 5 bins (> 124.5 cm): reward/tunnel responses
Landmarks at: [37, 65, 93, 120] cm
Using per-landmark window configuration:
  Landmark 1 at 37 cm: [12.0, 37.0] cm (-25, +0)
  Landmark 2 at 65 cm: [40.0, 65.0] cm (-25, +0)
  Landmark 3 at 93 cm: [68.0, 93.0] cm (-25, +0)
  Landmark 4 at 120 cm: [95.0, 120.0] cm (-25, +0)

=== VALIDATION SUMMARY ===
Total cells: 632
Valid cells with landmark preference: 287 (45.4%)

Rejection breakdown:
  - Zero activity: 0
  - Onset response (first 5 bins): 90
  - Reward/tunnel response (last 5 bins): 30
  - Peak outside landmark windows: 225

Landmark preference distribution:
  Landmark 1 (37 cm): 151 (52.6%)
  Landmark 2 (65 cm): 52 (18.1%)
  Landmark 3 (93 cm): 24 (8.4%)
  Landmark 4 (120 cm): 60 (20.9%)
  Day1__TSeries-07192026-0941-001


=== LANDMARK PREFERENCE IDENTIFICATION ===
Corridor: 0.5 to 129.5 cm (130 bins)
Bin spacing: 1.00 cm
Global peak exclusion:
  - First 5 bins (< 5.5 cm): onset responses
  - Last 5 bins (> 124.5 cm): reward/tunnel responses
Landmarks at: [37, 65, 93, 120] cm
Using per-landmark window configuration:
  Landmark 1 at 37 cm: [12.0, 37.0] cm (-25, +0)
  Landmark 2 at 65 cm: [40.0, 65.0] cm (-25, +0)
  Landmark 3 at 93 cm: [68.0, 93.0] cm (-25, +0)
  Landmark 4 at 120 cm: [95.0, 120.0] cm (-25, +0)

=== VALIDATION SUMMARY ===
Total cells: 603
Valid cells with landmark preference: 182 (30.2%)

Rejection breakdown:
  - Zero activity: 2
  - Onset response (first 5 bins): 169
  - Reward/tunnel response (last 5 bins): 32
  - Peak outside landmark windows: 218

Landmark preference distribution:
  Landmark 1 (37 cm): 85 (46.7%)
  Landmark 2 (65 cm): 54 (29.7%)
  Landmark 3 (93 cm): 14 (7.7%)
  Landmark 4 (120 cm): 29 (15.9%)
  Day3: 182/603 cells with a landmark preference

--- Day4 ---

=== LANDMA

,n_cells,n_preferring,fraction_preferring
Day1__TSeries-07192026-0941-001,632,287,0.454114
Day2,675,249,0.368889
Day3,603,182,0.301824
Day4,721,286,0.396671
Day5,700,218,0.311429
260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ,683,303,0.443631
260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE,578,254,0.439446
260726_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_2_DCZ,746,342,0.458445
260726_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_2_SALINE,537,288,0.536313
260728_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_3_DCZ,706,362,0.512748


## Diagnostic — SALINE vs DCZ, same pair, actual per-cell responses by landmark assignment

Function 6.1's summary table shows two consistent (5/5 pairs) patterns: total landmark-preferring fraction is lower under DCZ than its own paired saline, AND among the cells that still get assigned, Landmark 1 (37cm, closest to the onset-exclusion boundary) takes a bigger share under DCZ. Before merging `has_landmark_preference` onto Phase 4's SMI tables, check what's actually driving this with the same `plot_cells_by_landmark_assignment` visualization used for Function 6.0's single-session check -- this time on one full saline/DCZ pair side by side (DCZ1), one figure per session.

If DCZ's Landmark-1 cells still show tight, genuine peaks near 37cm (same as saline's), the L1-share inflation is more likely a real narrowing of coding toward the near landmark. If they look diffuse/onset-like compared to saline's Landmark-1 cells, that supports the artifact explanation instead. Either way, the question raised is a real one worth carrying forward: **if the cells DCZ knocks out of landmark-preferring status are specifically the ones encoding the farther landmarks (2/3/4), while the near-landmark response persists, that pattern itself would support a genuine reduction in spatial coding** (RSC silencing knocking out the more spatially-demanding far-landmark representations, leaving behind whatever the near-onset response actually is) -- which is exactly what the layer-specific composition check below is built to test.

In [132]:
# --- Diagnostic: SALINE vs DCZ landmark assignment, same pair (DCZ1) ---
saline_label = '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE'
dcz_label = '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ'

for label in (saline_label, dcz_label):
    tseries_dir = session_catalog[label]['tseries_dir']
    _, raw_results_pair = identify_session_landmark_preference(tseries_dir)

    preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
    preproc_data = files.read_h5(preproc_files[0])
    bin_centers_pair = preproc_data['bin_centers']

    print(f"\n{'='*30}  {label}  {'='*30}")
    fig = plot_cells_by_landmark_assignment(
        raw_results_pair, bin_centers_pair,
        landmark_positions=LANDMARK_POSITIONS,
        trim_start_bins=5, trim_end_bins=5,
    )
    fig.canvas.manager.set_window_title(label)
    plt.show()


=== LANDMARK PREFERENCE IDENTIFICATION ===
Corridor: 0.5 to 129.5 cm (130 bins)
Bin spacing: 1.00 cm
Global peak exclusion:
  - First 5 bins (< 5.5 cm): onset responses
  - Last 5 bins (> 124.5 cm): reward/tunnel responses
Landmarks at: [37, 65, 93, 120] cm
Using per-landmark window configuration:
  Landmark 1 at 37 cm: [12.0, 37.0] cm (-25, +0)
  Landmark 2 at 65 cm: [40.0, 65.0] cm (-25, +0)
  Landmark 3 at 93 cm: [68.0, 93.0] cm (-25, +0)
  Landmark 4 at 120 cm: [95.0, 120.0] cm (-25, +0)

=== VALIDATION SUMMARY ===
Total cells: 578
Valid cells with landmark preference: 254 (43.9%)

Rejection breakdown:
  - Zero activity: 2
  - Onset response (first 5 bins): 148
  - Reward/tunnel response (last 5 bins): 17
  - Peak outside landmark windows: 157

Landmark preference distribution:
  Landmark 1 (37 cm): 141 (55.5%)
  Landmark 2 (65 cm): 51 (20.1%)
  Landmark 3 (93 cm): 28 (11.0%)
  Landmark 4 (120 cm): 34 (13.4%)

==============================  260724_JSY_JSY090_LongitudinalImaging_D

## Setup — `load_all_group_dfs_from_phase4`

Reimplemented unchanged from Phase 5's Setup function. Loads every `{group}_comparison_table.csv` Phase 4's Function 4.8 already saved -- these already carry `session_label`, `condition`, `layer`, `SMI`, `valid`, `cell_idx` per row, which is exactly what Function 6.2 merges the landmark columns onto.

- **Input:** `output_dir` (the animal's `Phase4_SessionComparison_Results` folder).
- **Output:** `all_group_dfs` (`{group_name: df}`).

In [133]:
def load_all_group_dfs_from_phase4(output_dir):
    """
    Load every comparison group's table already saved by Phase 4's
    Function 4.8. See markdown above.

    Parameters
    ----------
    output_dir : str
        e.g. os.path.join(ANIMAL_DIR, 'Phase4_SessionComparison_Results').

    Returns
    -------
    all_group_dfs : dict
        {group_name: df}.
    """
    csv_paths = sorted(glob.glob(os.path.join(output_dir, '*_comparison_table.csv')))

    if not csv_paths:
        raise FileNotFoundError(f"No *_comparison_table.csv found in {output_dir} -- "
                                 "has Phase 4's save step (Function 4.8) been run for this animal?")

    all_group_dfs = {}
    for csv_path in csv_paths:
        group_name = os.path.basename(csv_path)[:-len('_comparison_table.csv')]
        df = pd.read_csv(csv_path)
        all_group_dfs[group_name] = df
        print(f"Loaded '{group_name}': {len(df)} cell-rows <- {csv_path}")

    print(f"\nLoaded {len(all_group_dfs)} group(s) from {output_dir}: {list(all_group_dfs.keys())}")
    return all_group_dfs


# --- Load the comparison tables Phase 4 already saved for this animal ---
OUTPUT_DIR = os.path.join(TEST_ANIMAL_DIR, 'Phase4_SessionComparison_Results')
all_group_dfs = load_all_group_dfs_from_phase4(OUTPUT_DIR)

Loaded 'Active_OL': 2158 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\Active_OL_comparison_table.csv
Loaded 'DCZ1': 1961 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\DCZ1_comparison_table.csv
Loaded 'DCZ2': 1983 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\DCZ2_comparison_table.csv
Loaded 'DCZ3': 1990 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\DCZ3_comparison_table.csv
Loaded 'Stationary_OL': 1934 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\Stationary_OL_comparison_table.csv
Loaded 'baseline': 3331 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\baseline_comparison_table.csv

Loaded 6 group(s) fro

## Function 6.2 — `merge_landmark_with_smi_table`

Attaches Function 6.1's landmark columns (`has_landmark_preference`, `preferred_landmark_position`, `preference_strength`) onto one of Phase 4's per-cell comparison tables, joining on `session_label` + `cell_idx` (both already present in Phase 4's saved tables and in every `landmark_df`). This is what makes `layer` + `condition` + `SMI` + `valid` (from Phase 4) and the landmark columns (from Phase 6) available together in one table -- needed both for the layer-specific landmark-composition check right below, and for every later SMI-by-landmark-preference comparison (6.3+).

- **Input:** `comparison_df` (one group's table, e.g. from `all_group_dfs['DCZ1']`), `landmark_lookup` (from Function 6.1).
- **Output:** `merged_df`.

In [134]:
def merge_landmark_with_smi_table(comparison_df, landmark_lookup):
    """
    Attach landmark columns onto one of Phase 4's per-cell comparison
    tables, by session_label + cell_idx. See markdown above.

    Parameters
    ----------
    comparison_df : pandas.DataFrame
        One group's table (e.g. all_group_dfs['DCZ1']).
    landmark_lookup : dict
        {session_label: landmark_df} from Function 6.1.

    Returns
    -------
    merged_df : pandas.DataFrame
        comparison_df with has_landmark_preference/preferred_landmark_position/
        preference_strength attached. Rows whose session_label has no
        matching landmark_lookup entry get has_landmark_preference=False
        (flagged via a printed warning) rather than silently NaN.
    """
    missing_labels = set(comparison_df['session_label'].unique()) - set(landmark_lookup.keys())
    if missing_labels:
        print(f"WARNING: {len(missing_labels)} session_label(s) in comparison_df have no landmark "
              f"data in landmark_lookup -- their rows will get has_landmark_preference=False: "
              f"{missing_labels}")

    landmark_parts = []
    for label, landmark_df in landmark_lookup.items():
        part = landmark_df.copy()
        part['session_label'] = label
        landmark_parts.append(part)
    all_landmark_df = pd.concat(landmark_parts, ignore_index=True)

    merged_df = comparison_df.merge(all_landmark_df, on=['session_label', 'cell_idx'], how='left')

    n_missing = merged_df['has_landmark_preference'].isna().sum()
    if n_missing > 0:
        print(f"  {n_missing}/{len(merged_df)} row(s) have no landmark match after merge "
              f"(session_label not in landmark_lookup) -- set to has_landmark_preference=False.")
    merged_df['has_landmark_preference'] = merged_df['has_landmark_preference'].fillna(False).astype(bool)

    return merged_df

In [135]:
# --- Test: merge landmark data onto the DCZ1 group's comparison table ---
merged_df = merge_landmark_with_smi_table(all_group_dfs['DCZ1'], landmark_lookup)
print(merged_df[['session_label', 'condition', 'layer', 'cell_idx', 'SMI', 'valid',
                  'has_landmark_preference', 'preferred_landmark_position']].head(10))

  session_label condition layer  cell_idx       SMI  valid  has_landmark_preference  preferred_landmark_position
0          Day5  baseline    L5         0 -0.332996  False                    False                          NaN
1          Day5  baseline    L5         1  1.000000  False                    False                          NaN
2          Day5  baseline    L5         2  0.901745   True                    False                          NaN
3          Day5  baseline    L5         3 -0.249370  False                    False                          NaN
4          Day5  baseline    L5         4  0.196909   True                     True                        120.0
5          Day5  baseline    L5         5  0.708991  False                    False                          NaN
6          Day5  baseline    L5         6 -0.035634   True                    False                          NaN
7          Day5  baseline    L5         7 -0.052388  False                    False             

## Diagnostic — `summarize_landmark_composition_by_layer` (layer-specific, as in Phase 5)

Directly tests the question raised above: is the saline→DCZ shift (lower landmark-preferring fraction overall, bigger Landmark-1 share among survivors) concentrated in particular layers, or uniform across all four -- same per-layer framing as Phase 5's `compare_smi_by_layer`, but on landmark *composition* (categorical: which landmark, or none) instead of SMI (continuous).

Per layer × condition: `n_cells`, `n_preferring`, `fraction_preferring`, and (among preferring cells only) the proportion assigned to each of the four landmarks. Plus a chi-square test per layer on the condition × landmark contingency table (preferring cells only) -- tests whether the *shape* of the landmark distribution differs between conditions within that layer, not just whether the preferring fraction differs.

If the farther landmarks (2/3/4) are the ones dropping out under DCZ specifically in deep layers (L5/L6, RSC's direct target) while superficial layers keep a more balanced distribution, that would support a genuine, layer-specific reduction in spatial (landmark) coding -- your hypothesis above, made testable per layer rather than just eyeballed from the whole-population numbers in Function 6.1's summary.

- **Input:** `merged_df` (from Function 6.2).
- **Output:** `composition_df` (one row per layer × condition), `chi2_results` (`{layer: {...} or None}`).

In [136]:
CANONICAL_LAYER_ORDER = ['L2/3', 'L4', 'L5', 'L6']


def _layer_order(layers_present):
    return ([l for l in CANONICAL_LAYER_ORDER if l in layers_present]
            + [l for l in layers_present if l not in CANONICAL_LAYER_ORDER])


def summarize_landmark_composition_by_layer(merged_df, layer_col='layer', condition_col='condition',
                                            preferred_landmark_col='preferred_landmark_position',
                                            has_pref_col='has_landmark_preference',
                                            landmark_positions=LANDMARK_POSITIONS):
    """
    Per layer x condition: n_cells, n_preferring, fraction_preferring, and
    (among preferring cells) the proportion assigned to each landmark. Plus
    a chi-square test per layer on the condition x landmark contingency
    table. See markdown above.

    Parameters
    ----------
    merged_df : pandas.DataFrame
        From merge_landmark_with_smi_table.
    layer_col, condition_col, preferred_landmark_col, has_pref_col : str
    landmark_positions : list of float

    Returns
    -------
    composition_df : pandas.DataFrame
        One row per (layer, condition).
    chi2_results : dict
        {layer: {'chi2', 'p', 'dof', 'contingency_table', 'conditions'} or None}.
    """
    layers_present = [l for l in merged_df[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    rows = []
    chi2_results = {}

    for layer in layer_order:
        layer_df = merged_df[merged_df[layer_col] == layer]
        conditions_present = [c for c in layer_df[condition_col].dropna().unique()]

        contingency_rows = []
        for cond in conditions_present:
            cond_df = layer_df[layer_df[condition_col] == cond]
            n_cells = len(cond_df)
            preferring = cond_df[cond_df[has_pref_col]]
            n_preferring = len(preferring)

            row = {
                'layer': layer, 'condition': cond,
                'n_cells': n_cells, 'n_preferring': n_preferring,
                'fraction_preferring': n_preferring / n_cells if n_cells > 0 else np.nan,
            }
            landmark_counts = []
            for i, lm_pos in enumerate(landmark_positions):
                n_lm = int((preferring[preferred_landmark_col] == lm_pos).sum())
                row[f'frac_L{i+1}'] = n_lm / n_preferring if n_preferring > 0 else np.nan
                landmark_counts.append(n_lm)
            rows.append(row)
            contingency_rows.append(landmark_counts)

        valid_rows = [(cond, counts) for cond, counts in zip(conditions_present, contingency_rows)
                      if sum(counts) > 0]
        if len(valid_rows) >= 2:
            table = np.array([counts for _, counts in valid_rows])
            try:
                chi2, p, dof, _ = chi2_contingency(table)
                chi2_results[layer] = {'chi2': chi2, 'p': p, 'dof': dof,
                                       'contingency_table': table,
                                       'conditions': [c for c, _ in valid_rows]}
            except ValueError as e:
                print(f"  {layer}: chi-square skipped ({e})")
                chi2_results[layer] = None
        else:
            chi2_results[layer] = None

    composition_df = pd.DataFrame(rows)
    print(composition_df.to_string(index=False))

    print("\nChi-square (condition x landmark distribution, preferring cells only):")
    for layer, result in chi2_results.items():
        if result is not None:
            print(f"  {layer}: chi2={result['chi2']:.3f}, p={result['p']:.4f}, dof={result['dof']} "
                  f"(conditions: {result['conditions']})")
        else:
            print(f"  {layer}: skipped (not enough data)")

    return composition_df, chi2_results

## Diagnostic — `plot_landmark_composition_by_layer`

Grid (one panel per layer) of stacked bars, one bar per condition, segments = proportion of preferring cells assigned to each of the four landmarks (same style as Phase 5's per-layer grid plots). `fraction_preferring` is annotated above each bar, since it's a separate axis from composition (a bar can have a very different color mix while its total preferring fraction stays the same, or vice versa).

- **Input:** `composition_df` (from `summarize_landmark_composition_by_layer`).
- **Output:** `fig`.

In [137]:
def plot_landmark_composition_by_layer(composition_df, landmark_positions=LANDMARK_POSITIONS, title=''):
    """
    Grid of stacked bars (one panel per layer, one bar per condition --
    saline/dcz only, baseline excluded -- segments = landmark proportions
    among preferring cells). See markdown above.

    Each bar is annotated with Q1_pref/Q{n}_pref -- the population-level
    percentage of ALL cells (not just preferring ones) preferring the
    first/last landmark specifically (frac_L{i} * fraction_preferring),
    replacing the earlier ambiguous 'pref=X%' (which was fraction_preferring
    -- ANY landmark, not landmark 1).
    """
    layer_order = _layer_order(composition_df['layer'].unique())
    landmark_cols = [f'frac_L{i+1}' for i in range(len(landmark_positions))]
    colors = plt.cm.viridis(np.linspace(0, 1, len(landmark_positions)))
    condition_order = ['saline', 'dcz']
    n_landmarks = len(landmark_positions)

    fig, axes = plt.subplots(1, len(layer_order), figsize=(4.5 * len(layer_order), 6.5), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, layer in zip(axes, layer_order):
        layer_rows = composition_df[composition_df['layer'] == layer]
        conditions = [c for c in condition_order if c in layer_rows['condition'].values]

        bottoms = np.zeros(len(conditions))
        for i, col in enumerate(landmark_cols):
            vals = np.array([layer_rows.loc[layer_rows['condition'] == c, col].values[0]
                             if c in layer_rows['condition'].values else 0 for c in conditions])
            vals = np.nan_to_num(vals)
            ax.bar(conditions, vals, bottom=bottoms, color=colors[i],
                   label=f'{landmark_positions[i]}cm' if ax is axes[0] else None)
            bottoms += vals

        for x, c in enumerate(conditions):
            row = layer_rows.loc[layer_rows['condition'] == c]
            if len(row) == 0:
                continue
            frac_pref = row['fraction_preferring'].values[0]
            n_cells = int(row['n_cells'].values[0])
            q1_pref = row['frac_L1'].values[0] * frac_pref
            q4_pref = row[f'frac_L{n_landmarks}'].values[0] * frac_pref
            ax.text(x, 1.03, f'Q1_pref={q1_pref:.0%}\nQ{n_landmarks}_pref={q4_pref:.0%}\n(n={n_cells})',
                   ha='center', fontsize=9)

        ax.set_ylim(0, 1.25)
        ax.set_title(layer)

    axes[0].set_ylabel('Proportion of preferring cells\nby landmark')
    axes[0].legend(fontsize=9, loc='upper left', bbox_to_anchor=(0, -0.08), ncol=len(landmark_positions))
    fig.suptitle(title, fontsize=18, fontweight='bold')
    plt.tight_layout()
    return fig

In [138]:
# --- Test: layer-specific landmark composition, DCZ1 group ---
composition_df, chi2_results = summarize_landmark_composition_by_layer(merged_df)

fig = plot_landmark_composition_by_layer(composition_df, title='DCZ1 -- landmark composition by layer')
plt.show()

layer condition  n_cells  n_preferring  fraction_preferring  frac_L1  frac_L2  frac_L3  frac_L4
 L2/3  baseline      110            44             0.400000 0.613636 0.181818 0.113636 0.090909
 L2/3       dcz      127            65             0.511811 0.553846 0.169231 0.092308 0.184615
 L2/3    saline       96            57             0.593750 0.543860 0.298246 0.087719 0.070175
   L4  baseline      183            63             0.344262 0.698413 0.126984 0.031746 0.142857
   L4       dcz      158            89             0.563291 0.674157 0.168539 0.011236 0.146067
   L4    saline      133            72             0.541353 0.569444 0.236111 0.055556 0.138889
   L5  baseline      239            75             0.313808 0.613333 0.146667 0.053333 0.186667
   L5       dcz      232           101             0.435345 0.633663 0.178218 0.029703 0.158416
   L5    saline      203            83             0.408867 0.554217 0.156627 0.168675 0.120482
   L6  baseline      168            36  

## Driver — `run_landmark_composition_analysis_all_groups`

Loops the merge (6.2) + layer composition (`summarize_landmark_composition_by_layer`/`plot_landmark_composition_by_layer`) over every comparison group, so DCZ1's pattern (fraction_preferring drops saline→dcz in all 4 layers, but the significant chi-squares in L2/3/L4/L5 are actually driven by **saline** having an anomalously elevated far-landmark (`frac_L4`) share relative to *both* baseline and dcz, not by dcz being suppressed relative to baseline) can be checked for replication across DCZ2/DCZ3/Active_OL/Stationary_OL. Skips the single-condition `baseline` group (no condition contrast to test), same convention as Phase 5's `run_layer_analysis_all_groups`.

- **Input:** `all_group_dfs` (from `load_all_group_dfs_from_phase4`), `landmark_lookup` (from Function 6.1).
- **Output:** `results` (`{group_name: {'merged_df', 'composition_df', 'chi2_results', 'fig'}}`).

In [139]:
def run_landmark_composition_analysis_for_group(comparison_df, landmark_lookup, group_name=''):
    """
    Merge landmark data onto one group's table, then run
    summarize_landmark_composition_by_layer + plot_landmark_composition_by_layer.
    See markdown above.

    Parameters
    ----------
    comparison_df : pandas.DataFrame
        One group's table (e.g. all_group_dfs['DCZ2']).
    landmark_lookup : dict
        {session_label: landmark_df} from Function 6.1.
    group_name : str

    Returns
    -------
    result : dict
        {'merged_df', 'composition_df', 'chi2_results', 'fig'}.
    """
    print(f"\n{'='*90}\nLandmark composition analysis: {group_name}\n{'='*90}")

    merged_df = merge_landmark_with_smi_table(comparison_df, landmark_lookup)
    composition_df, chi2_results = summarize_landmark_composition_by_layer(merged_df)
    fig = plot_landmark_composition_by_layer(composition_df, title=f'{group_name} -- landmark composition by layer')
    plt.show()

    return {
        'merged_df': merged_df,
        'composition_df': composition_df,
        'chi2_results': chi2_results,
        'fig': fig,
    }


def run_landmark_composition_analysis_all_groups(all_group_dfs, landmark_lookup, condition_col='condition'):
    """
    Loop run_landmark_composition_analysis_for_group over every group,
    skipping single-condition groups (e.g. baseline -- no contrast to
    test). See markdown above.

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from load_all_group_dfs_from_phase4.
    landmark_lookup : dict
        {session_label: landmark_df} from Function 6.1.
    condition_col : str

    Returns
    -------
    results : dict
        {group_name: run_landmark_composition_analysis_for_group(...) result}.
    """
    results = {}
    for group_name, df in all_group_dfs.items():
        conditions_present = df[condition_col].dropna().unique()
        if len(conditions_present) < 2:
            print(f"\n{'='*90}\n{group_name}: only {len(conditions_present)} condition(s) present "
                  f"({list(conditions_present)}) -- skipping (no contrast to test).\n{'='*90}")
            continue
        results[group_name] = run_landmark_composition_analysis_for_group(
            df, landmark_lookup, group_name=group_name)

    return results

In [140]:
# --- Run across all groups ---
landmark_composition_results = run_landmark_composition_analysis_all_groups(all_group_dfs, landmark_lookup)

# --- Replication check: is saline's frac_L4 elevation (seen in DCZ1) consistent
#     across the other groups too, or specific to DCZ1? ---
rows = []
for group_name, result in landmark_composition_results.items():
    comp = result['composition_df'].set_index(['layer', 'condition'])
    for layer in comp.index.get_level_values('layer').unique():
        row = {'group': group_name, 'layer': layer}
        for cond in ('baseline', 'saline', 'dcz'):
            row[f'{cond}_frac_L4'] = comp['frac_L4'].get((layer, cond), np.nan)
            row[f'{cond}_fraction_preferring'] = comp['fraction_preferring'].get((layer, cond), np.nan)
        rows.append(row)

cross_group_summary = pd.DataFrame(rows)
pd.set_option('display.width', 150)
cross_group_summary


Landmark composition analysis: Active_OL
layer condition  n_cells  n_preferring  fraction_preferring  frac_L1  frac_L2  frac_L3  frac_L4
 L2/3  baseline      110            44             0.400000 0.613636 0.181818 0.113636 0.090909
 L2/3       dcz      105            58             0.552381 0.913793 0.017241 0.017241 0.051724
 L2/3    saline      106            78             0.735849 0.717949 0.115385 0.051282 0.115385
   L4  baseline      183            63             0.344262 0.698413 0.126984 0.031746 0.142857
   L4       dcz      237           118             0.497890 0.898305 0.033898 0.008475 0.059322
   L4    saline      168            99             0.589286 0.747475 0.121212 0.070707 0.060606
   L5  baseline      239            75             0.313808 0.613333 0.146667 0.053333 0.186667
   L5       dcz      281           138             0.491103 0.898551 0.065217 0.000000 0.036232
   L5    saline      213           121             0.568075 0.719008 0.140496 0.049587 0.09090

,group,layer,baseline_frac_L4,baseline_fraction_preferring,saline_frac_L4,saline_fraction_preferring,dcz_frac_L4,dcz_fraction_preferring
0,Active_OL,L2/3,0.090909,0.400000,0.115385,0.735849,0.051724,0.552381
1,Active_OL,L4,0.142857,0.344262,0.060606,0.589286,0.059322,0.497890
2,Active_OL,L5,0.186667,0.313808,0.090909,0.568075,0.036232,0.491103
3,Active_OL,L6,0.388889,0.214286,0.133333,0.323741,0.130435,0.330144
4,DCZ1,L2/3,0.090909,0.400000,0.070175,0.593750,0.184615,0.511811
5,DCZ1,L4,0.142857,0.344262,0.138889,0.541353,0.146067,0.563291
6,DCZ1,L5,0.186667,0.313808,0.120482,0.408867,0.158416,0.435345
7,DCZ1,L6,0.388889,0.214286,0.238095,0.287671,0.437500,0.289157
8,DCZ2,L2/3,0.090909,0.400000,0.071429,0.691358,0.000000,0.446281
9,DCZ2,L4,0.142857,0.344262,0.090909,0.682171,0.103448,0.633880


## Figures — last-landmark (120cm) preference reduction under DCZ

Two figures demonstrating the saline→dcz drop in the fraction of cells preferring the last landmark, using data already in `landmark_composition_results` (no new computation of landmark identity, just a different aggregation/view of it):

1. **Population-level paired slope plot** — one line per comparison group (DCZ1/DCZ2/DCZ3/Active_OL/Stationary_OL), saline→dcz, y-axis = fraction of *all* recorded cells (not just landmark-preferring ones) whose preferred landmark is the last one. Baseline (the single shared Day5 reference) drawn as one horizontal line.
2. **Per-layer small multiples** — same slope plot, one panel per layer, to show whether the drop holds up within each layer individually.

Both use `overall_frac_L{n}` = `fraction_preferring x frac_L{n}` -- the fraction of the *entire* cell population assigned to the last landmark, not just the fraction among the shrinking pool of landmark-preferring cells (`composition_df`'s `frac_L4`) -- since "reduction in last-landmark-preferring cells" is a population-level claim.

A paired Wilcoxon signed-rank test (n=5 pairs) backs the population-level figure numerically.

**Caveat carried over from the last two turns**: this is still the cross-session, trial-count-confounded comparison (saline sessions are shorter than their paired dcz sessions in every pair) -- these figures demonstrate the pattern clearly, they don't yet resolve whether it's a trial-count artifact or a real effect. That's still what the within-session SMI check (Option B) is for.

- **Input:** `landmark_composition_results` (from the driver run above).
- **Output:** `pop_df`, two figures, a printed Wilcoxon test result.

In [141]:
# Fixed categorical color per comparison group -- never cycled/re-painted,
# same group always gets the same color across every figure in this notebook.
GROUP_COLORS = {
    'DCZ1': '#1b9e77',
    'DCZ2': '#d95f02',
    'DCZ3': '#7570b3',
    'Active_OL': '#e7298a',
    'Stationary_OL': '#66a61e',
}


def compute_population_landmark_fraction(landmark_composition_results, landmark_positions=LANDMARK_POSITIONS):
    """
    Population-level (all layers pooled, all cells -- not just preferring
    ones) fraction of cells preferring each landmark, per group x
    condition. See markdown above for why this differs from
    composition_df's frac_L{n} (which is out of preferring cells only).

    Parameters
    ----------
    landmark_composition_results : dict
        {group_name: run_landmark_composition_analysis_for_group(...) result}.
    landmark_positions : list of float

    Returns
    -------
    pop_df : pandas.DataFrame
        One row per (group, condition): n_cells, n_preferring,
        fraction_preferring, overall_frac_L1..overall_frac_L{n}.
    """
    rows = []
    for group_name, result in landmark_composition_results.items():
        merged_df = result['merged_df']
        for cond in merged_df['condition'].dropna().unique():
            cond_df = merged_df[merged_df['condition'] == cond]
            n_cells = len(cond_df)
            preferring = cond_df[cond_df['has_landmark_preference']]
            n_preferring = len(preferring)

            row = {
                'group': group_name, 'condition': cond,
                'n_cells': n_cells, 'n_preferring': n_preferring,
                'fraction_preferring': n_preferring / n_cells if n_cells > 0 else np.nan,
            }
            for i, lm_pos in enumerate(landmark_positions):
                n_lm = int((preferring['preferred_landmark_position'] == lm_pos).sum())
                row[f'overall_frac_L{i+1}'] = n_lm / n_cells if n_cells > 0 else np.nan
            rows.append(row)

    pop_df = pd.DataFrame(rows)
    print(pop_df.to_string(index=False))
    return pop_df

In [142]:
def plot_landmark_last_slope_population(pop_df, landmark_positions=LANDMARK_POSITIONS):
    """
    Paired slope plot: one line per comparison group, saline -> dcz,
    population-level fraction of ALL cells preferring the last landmark.
    Baseline shown as a single horizontal reference line (all groups
    share the same baseline session -- see markdown above). See markdown
    above for the full design rationale.

    Parameters
    ----------
    pop_df : pandas.DataFrame
        From compute_population_landmark_fraction.
    landmark_positions : list of float

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    metric_col = f'overall_frac_L{len(landmark_positions)}'
    fig, ax = plt.subplots(figsize=(7.5, 8.5))

    baseline_val = pop_df.loc[pop_df['condition'] == 'baseline', metric_col].mean()
    ax.axhline(baseline_val, color='tab:blue', linestyle='--', linewidth=2, alpha=0.7,
               zorder=1, label=f'baseline ({baseline_val:.1%})')

    x_positions = {'saline': 0, 'dcz': 1}
    for group_name in pop_df['group'].unique():
        group_rows = pop_df[pop_df['group'] == group_name]
        color = GROUP_COLORS.get(group_name, 'gray')
        xs, ys = [], []
        for cond in ('saline', 'dcz'):
            val = group_rows.loc[group_rows['condition'] == cond, metric_col]
            if len(val) > 0:
                xs.append(x_positions[cond])
                ys.append(val.values[0])
        if len(xs) == 2:
            ax.plot(xs, ys, color=color, marker='o', markersize=11, linewidth=2.5,
                    solid_capstyle='round', zorder=3)
            ax.text(xs[-1] + 0.05, ys[-1], group_name, color=color, fontsize=14,
                    va='center', ha='left', fontweight='bold')

    ax.set_xlim(-0.3, 1.6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['saline', 'dcz'])
    ax.set_ylabel(f'Fraction of all cells preferring\nlandmark {len(landmark_positions)} '
                  f'({landmark_positions[-1]:.0f}cm)')
    ax.legend(loc='upper right', fontsize=13, frameon=False)
    ax.set_title('Last-landmark preference drops under DCZ\nin every saline/DCZ pair (JSY093)')
    plt.tight_layout()
    return fig

In [143]:
def plot_landmark_last_slope_by_layer(landmark_composition_results, landmark_positions=LANDMARK_POSITIONS):
    """
    Small multiples (one panel per layer) of the same paired slope plot,
    using composition_df's per-layer frac_L{n} * fraction_preferring as
    the population-level (out of all cells in that layer) metric. See
    markdown above.

    Parameters
    ----------
    landmark_composition_results : dict
        {group_name: run_landmark_composition_analysis_for_group(...) result}.
    landmark_positions : list of float

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    n_landmarks = len(landmark_positions)
    last_col = f'frac_L{n_landmarks}'

    first_result = next(iter(landmark_composition_results.values()))
    layer_order = _layer_order(first_result['composition_df']['layer'].unique())

    fig, axes = plt.subplots(1, len(layer_order), figsize=(5 * len(layer_order), 7.5), sharey=True)
    axes = np.atleast_1d(axes)
    x_positions = {'saline': 0, 'dcz': 1}

    for ax, layer in zip(axes, layer_order):
        baseline_vals = []
        for group_name, result in landmark_composition_results.items():
            comp = result['composition_df']
            layer_rows = comp[comp['layer'] == layer].set_index('condition')

            if 'baseline' in layer_rows.index:
                b = layer_rows.loc['baseline']
                baseline_vals.append(b['fraction_preferring'] * b[last_col])

            color = GROUP_COLORS.get(group_name, 'gray')
            xs, ys = [], []
            for cond in ('saline', 'dcz'):
                if cond in layer_rows.index:
                    row = layer_rows.loc[cond]
                    xs.append(x_positions[cond])
                    ys.append(row['fraction_preferring'] * row[last_col])
            if len(xs) == 2:
                ax.plot(xs, ys, color=color, marker='o', markersize=8, linewidth=2,
                        solid_capstyle='round', zorder=3)

        if baseline_vals:
            ax.axhline(np.mean(baseline_vals), color='tab:blue', linestyle='--',
                       linewidth=1.5, alpha=0.7, zorder=1)

        ax.set_xlim(-0.3, 1.3)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['saline', 'dcz'])
        ax.set_title(layer)

    axes[0].set_ylabel(f'Fraction of cells preferring\nlandmark {n_landmarks} ({landmark_positions[-1]:.0f}cm)')

    handles = [plt.Line2D([0], [0], color=GROUP_COLORS.get(g, 'gray'), marker='o', linewidth=2, label=g)
              for g in landmark_composition_results.keys()]
    handles.append(plt.Line2D([0], [0], color='tab:blue', linestyle='--', label='baseline'))
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), fontsize=11,
              bbox_to_anchor=(0.5, -0.06), frameon=False)

    fig.suptitle('Last-landmark preference drop under DCZ, by layer (JSY093)')
    plt.tight_layout()
    return fig

In [144]:
# --- Build the figures ---
pop_df = compute_population_landmark_fraction(landmark_composition_results)

fig1 = plot_landmark_last_slope_population(pop_df)
plt.show()

fig2 = plot_landmark_last_slope_by_layer(landmark_composition_results)
plt.show()

# --- Paired Wilcoxon signed-rank test: saline vs dcz, population-level, n=5 pairs ---
last_col = f'overall_frac_L{len(LANDMARK_POSITIONS)}'
saline_vals = pop_df.loc[pop_df['condition'] == 'saline'].set_index('group')[last_col]
dcz_vals = pop_df.loc[pop_df['condition'] == 'dcz'].set_index('group')[last_col]
paired = pd.DataFrame({'saline': saline_vals, 'dcz': dcz_vals}).dropna()
paired['drop'] = paired['saline'] - paired['dcz']

stat, p = wilcoxon(paired['saline'], paired['dcz'])
print(f"\nPaired comparison (saline vs dcz, population-level, n={len(paired)} pairs):")
print(paired.to_string())
print(f"\nWilcoxon signed-rank: stat={stat:.3f}, p={p:.4f}")

        group condition  n_cells  n_preferring  fraction_preferring  overall_frac_L1  overall_frac_L2  overall_frac_L3  overall_frac_L4
    Active_OL  baseline      700           218             0.311429         0.190000         0.044286         0.018571         0.058571
    Active_OL       dcz      832           383             0.460337         0.408654         0.020433         0.002404         0.028846
    Active_OL    saline      626           343             0.547923         0.386581         0.075080         0.035144         0.051118
         DCZ1  baseline      700           218             0.311429         0.190000         0.044286         0.018571         0.058571
         DCZ1       dcz      683           303             0.443631         0.267936         0.068814         0.016105         0.090776
         DCZ1    saline      578           254             0.439446         0.243945         0.088235         0.048443         0.058824
         DCZ2  baseline      700           218  

## Diagnostic — open-loop vs. closed-loop: does the saline→dcz drop differ, and is it layer-dependent?

Open-loop VR (Active_OL, Stationary_OL) decouples self-motion from visual flow, unlike the closed-loop DCZ1/DCZ2/DCZ3 sessions -- RSC is thought to integrate exactly that kind of self-motion/vision coupling for navigation, so it's worth checking whether the saline→dcz drop in last-landmark preference differs in size between open- and closed-loop, and whether any layer behaves differently under open-loop specifically.

No new landmark computation -- this pools `composition_df`'s per-layer `frac_L{n} x fraction_preferring x n_cells` (already in `landmark_composition_results`) into one population-level saline/dcz estimate per group, and flags any layer where the direction reverses (dcz > saline instead of the usual saline > dcz).

**Same caveat as everything upstream of Option B**: this is still the cross-session, trial-count-confounded comparison. A difference in effect size between open- and closed-loop groups could reflect a real open-loop-specific effect, or could just be trial-count noise -- it isn't resolved by this cell either way.

- **Input:** `landmark_composition_results`.
- **Output:** `openloop_effect_summary` (one row per group: population-level saline/dcz far-landmark fraction, absolute/relative drop, list of any layer(s) where the direction reverses).

In [145]:
def summarize_openloop_vs_closedloop_effect(landmark_composition_results, landmark_positions=LANDMARK_POSITIONS):
    """
    Population-level (per-layer, n_cells-weighted) saline vs dcz far-landmark
    fraction per group, plus which layer(s), if any, reverse direction
    (dcz > saline). See markdown above.

    Parameters
    ----------
    landmark_composition_results : dict
        {group_name: run_landmark_composition_analysis_for_group(...) result}.
    landmark_positions : list of float

    Returns
    -------
    summary_df : pandas.DataFrame
        One row per group: saline_pop_frac_last, dcz_pop_frac_last,
        absolute_drop, relative_drop, reversed_layers.
    """
    last_col = f'frac_L{len(landmark_positions)}'
    rows = []

    for group_name, result in landmark_composition_results.items():
        comp = result['composition_df']
        pop = {}
        for cond in ('saline', 'dcz'):
            cond_rows = comp[comp['condition'] == cond]
            n_last_total = (cond_rows[last_col] * cond_rows['fraction_preferring'] * cond_rows['n_cells']).sum()
            n_cells_total = cond_rows['n_cells'].sum()
            pop[cond] = n_last_total / n_cells_total if n_cells_total > 0 else np.nan

        reversed_layers = []
        for layer in comp['layer'].unique():
            layer_rows = comp[comp['layer'] == layer].set_index('condition')
            if 'saline' in layer_rows.index and 'dcz' in layer_rows.index:
                if layer_rows.loc['dcz', last_col] > layer_rows.loc['saline', last_col]:
                    reversed_layers.append(layer)

        rows.append({
            'group': group_name,
            'saline_pop_frac_last': pop['saline'],
            'dcz_pop_frac_last': pop['dcz'],
            'absolute_drop': pop['saline'] - pop['dcz'],
            'relative_drop': (pop['saline'] - pop['dcz']) / pop['saline'] if pop['saline'] > 0 else np.nan,
            'reversed_layers': reversed_layers,
        })

    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False))
    return summary_df


# --- Run it ---
openloop_effect_summary = summarize_openloop_vs_closedloop_effect(landmark_composition_results)

        group  saline_pop_frac_last  dcz_pop_frac_last  absolute_drop  relative_drop    reversed_layers
    Active_OL              0.051118           0.028846       0.022272       0.435697                 []
         DCZ1              0.058824           0.090776      -0.031952      -0.543192 [L2/3, L4, L5, L6]
         DCZ2              0.050279           0.056300      -0.006021      -0.119750       [L4, L5, L6]
         DCZ3              0.041096           0.109065      -0.067969      -1.653919 [L2/3, L4, L5, L6]
Stationary_OL              0.032986           0.016717       0.016269       0.493201                 []


## Option B — within-session SMI: does landmark-4 preference actually mean "more spatial"?

Every comparison so far has been cross-session (saline vs. its paired dcz session), which is exactly where the trial-count confound lives (trial count differs *between* sessions, is fixed *within* one). This test sidesteps that entirely: **within a single session**, compare SMI (already computed by Phase 3, same trial count for every cell in that session by definition) between cells preferring the first landmark (37cm, the onset-adjacent one) and cells preferring the last landmark (120cm). If the last-landmark group has higher SMI consistently across sessions -- regardless of condition -- that validates treating far-landmark preference as a genuine spatial-coding marker before trusting any cross-condition comparison built on it.

Steps:
1. `build_all_sessions_landmark_smi_table` -- collapses `landmark_composition_results`' per-group `merged_df`s (which redundantly repeat the shared baseline session across every group) back down to one row per unique (session, cell) across all 15 sessions.
2. `compare_smi_landmark1_vs_last` -- one session's within-session Mann-Whitney U, landmark-1-preferring vs landmark-last-preferring cells, `valid`-filtered.
3. `run_landmark1_vs_last_smi_analysis` -- loops (2) over every session, builds a summary table, and a paired Wilcoxon test across sessions on the median-SMI difference (first vs last), to check whether the direction is consistent regardless of condition.
4. A slope plot -- one line per session, colored by condition (not group this time, since the point here is condition-independence), x = landmark1 vs landmark-last, y = median SMI.

- **Input:** `landmark_composition_results`.
- **Output:** `all_sessions_df`, `landmark_smi_results`, `landmark_smi_summary_df`, `fig`.

In [146]:
def build_all_sessions_landmark_smi_table(landmark_composition_results):
    """
    Concatenate every group's merged_df and drop duplicate
    (session_label, cell_idx) rows -- since the shared baseline (Day5)
    appears identically in every group's merged_df, this collapses back
    down to one row per unique (session, cell) across all sessions
    discovered for this animal. See markdown above.

    Parameters
    ----------
    landmark_composition_results : dict
        {group_name: run_landmark_composition_analysis_for_group(...) result}.

    Returns
    -------
    all_sessions_df : pandas.DataFrame
    """
    all_df = pd.concat([r['merged_df'] for r in landmark_composition_results.values()], ignore_index=True)
    all_df = all_df.drop_duplicates(subset=['session_label', 'cell_idx']).reset_index(drop=True)
    print(f"{len(all_df)} cell-rows across {all_df['session_label'].nunique()} unique sessions.")
    print(all_df.groupby('session_label')['condition'].first().to_string())
    return all_df

In [147]:
def compare_smi_landmark1_vs_last(session_df, filter_col='valid',
                                  landmark_col='preferred_landmark_position', value_col='SMI',
                                  landmark_positions=LANDMARK_POSITIONS):
    """
    Within one session, Mann-Whitney U comparing SMI between cells
    preferring the first landmark vs cells preferring the last landmark
    only (not all 4 -- more power per group, and it's the direct
    near-vs-far contrast the hypothesis is about). See markdown above.

    Parameters
    ----------
    session_df : pandas.DataFrame
        One session's rows only (same trial count for every row).
    filter_col, landmark_col, value_col : str
    landmark_positions : list of float

    Returns
    -------
    result : dict or None
        None if either group has fewer than 2 cells. Otherwise:
        {'n_first', 'n_last', 'median_SMI_first', 'median_SMI_last',
         'U_stat', 'p'}.
    """
    first_lm, last_lm = landmark_positions[0], landmark_positions[-1]
    d = session_df[session_df[filter_col] & session_df['has_landmark_preference']
                   & session_df[landmark_col].isin([first_lm, last_lm])]

    smi_first = d.loc[d[landmark_col] == first_lm, value_col].to_numpy()
    smi_last = d.loc[d[landmark_col] == last_lm, value_col].to_numpy()

    if len(smi_first) < 2 or len(smi_last) < 2:
        return None

    u_stat, p = mannwhitneyu(smi_first, smi_last, alternative='two-sided')

    return {
        'n_first': len(smi_first), 'n_last': len(smi_last),
        'median_SMI_first': float(np.median(smi_first)),
        'median_SMI_last': float(np.median(smi_last)),
        'U_stat': u_stat, 'p': p,
    }


def run_landmark1_vs_last_smi_analysis(all_sessions_df):
    """
    Loops compare_smi_landmark1_vs_last over every session, builds a
    summary table, and a paired Wilcoxon test across sessions on the
    median-SMI difference (last - first) -- tests whether the direction
    is consistent regardless of condition. See markdown above.

    Parameters
    ----------
    all_sessions_df : pandas.DataFrame
        From build_all_sessions_landmark_smi_table.

    Returns
    -------
    results : dict
        {session_label: compare_smi_landmark1_vs_last(...) result or None}.
    summary_df : pandas.DataFrame
        One row per session with a valid comparison: session_label,
        condition, n_first, n_last, median_SMI_first, median_SMI_last,
        SMI_diff_last_minus_first, p.
    """
    results = {}
    summary_rows = []
    for label, session_df in all_sessions_df.groupby('session_label'):
        result = compare_smi_landmark1_vs_last(session_df)
        results[label] = result
        if result is None:
            print(f"{label}: skipped (fewer than 2 cells in landmark 1 or landmark-last group)")
            continue
        summary_rows.append({
            'session_label': label,
            'condition': session_df['condition'].iloc[0],
            'n_first': result['n_first'], 'n_last': result['n_last'],
            'median_SMI_first': result['median_SMI_first'],
            'median_SMI_last': result['median_SMI_last'],
            'SMI_diff_last_minus_first': result['median_SMI_last'] - result['median_SMI_first'],
            'p': result['p'],
        })

    summary_df = pd.DataFrame(summary_rows)
    print(f"\n{summary_df.to_string(index=False)}")

    n_positive = (summary_df['SMI_diff_last_minus_first'] > 0).sum()
    print(f"\n{n_positive}/{len(summary_df)} sessions show higher median SMI in "
          f"landmark-last-preferring cells than landmark-1-preferring cells "
          f"(regardless of condition).")

    if len(summary_df) >= 2:
        stat, p = wilcoxon(summary_df['median_SMI_first'], summary_df['median_SMI_last'])
        print(f"\nPaired Wilcoxon signed-rank across sessions (median_SMI_first vs "
              f"median_SMI_last, n={len(summary_df)} sessions): stat={stat:.3f}, p={p:.4f}")

    return results, summary_df

In [148]:
def plot_smi_landmark1_vs_last(summary_df, landmark_positions=LANDMARK_POSITIONS):
    """
    Slope plot: one line per session, x = landmark1-preferring vs
    landmark-last-preferring cells, y = median SMI within that session.
    Colored by condition (not group) -- the point here is whether the
    direction holds regardless of condition, since each line is already
    trial-count-free by construction (same session on both ends).

    Parameters
    ----------
    summary_df : pandas.DataFrame
        From run_landmark1_vs_last_smi_analysis.
    landmark_positions : list of float

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    color_by_condition = {'baseline': 'tab:blue', 'saline': 'tab:orange', 'dcz': 'tab:green'}
    fig, ax = plt.subplots(figsize=(7, 8.5))

    x_positions = {'first': 0, 'last': 1}
    for _, row in summary_df.iterrows():
        color = color_by_condition.get(row['condition'], 'gray')
        ax.plot([x_positions['first'], x_positions['last']],
                [row['median_SMI_first'], row['median_SMI_last']],
                color=color, marker='o', markersize=8, linewidth=1.8, alpha=0.75, zorder=3)

    handles = [plt.Line2D([0], [0], color=c, marker='o', linewidth=2, label=cond)
              for cond, c in color_by_condition.items() if cond in summary_df['condition'].values]
    ax.legend(handles=handles, loc='upper left', fontsize=13, frameon=False)

    ax.set_xlim(-0.3, 1.3)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f'landmark 1\n({landmark_positions[0]:.0f}cm)',
                        f'landmark {len(landmark_positions)}\n({landmark_positions[-1]:.0f}cm)'])
    ax.set_ylabel('Median SMI (within-session)')
    ax.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.set_title('Within-session SMI: landmark-1- vs\nlandmark-last-preferring cells (JSY093)')
    plt.tight_layout()
    return fig

In [149]:
# --- Run Option B ---
all_sessions_df = build_all_sessions_landmark_smi_table(landmark_composition_results)

landmark_smi_results, landmark_smi_summary_df = run_landmark1_vs_last_smi_analysis(all_sessions_df)

fig = plot_smi_landmark1_vs_last(landmark_smi_summary_df)
plt.show()

7226 cell-rows across 11 unique sessions.
session_label
260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ                             dcz
260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE                       saline
260726_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_2_DCZ                             dcz
260726_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_2_SALINE                       saline
260728_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_3_DCZ                             dcz
260728_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_3_SALINE                       saline
260730_JSY_JSY090_LongitudinalImaging_DREADD_ActiveOpenLoop_Saline_DCZ_DCZ                dcz
260730_JSY_JSY090_LongitudinalImaging_DREADD_ActiveOpenLoop_Saline_DCZ_SALINE          saline
260801_JSY_JSY090_LongitudinalImaging_DREADD_StationaryOpenLoop_Saline_DCZ_DCZ            dcz
260801_JSY_JSY090_LongitudinalImaging_DREADD_StationaryOpenLoop_Saline_DCZ_SALINE      saline
Day5

## Diagnostic — `test_condition_effect_on_landmark_metric` (re-added)

Formally tests whether `session_type` (baseline/saline/dcz) predicts `fraction_preferring` (and `frac_L4`, as a formal cross-check of the already-strong descriptive evidence) beyond what `n_trials` alone explains. Weighted least squares: `metric ~ n_trials + C(session_type)`, weighted by each metric's own denominator (`n_cells` for `fraction_preferring`, `n_preferring` for `frac_L4`). Same design as Phase 4's Function 4.6 (`test_condition_effect_on_reliability`), reimplemented here.

Uses all 15 discovered sessions (not just the 11 in `landmark_composition_results`/`all_sessions_df`, which only include the single shared Day5 baseline) -- `landmark_lookup` + `session_catalog` have every baseline day (1-5), giving more statistical power for this specific check.

**What to look for when reading the output**: for each metric, the `C(session_type)[T.dcz]` and `C(session_type)[T.saline]` rows (relative to the `baseline` reference level) -- if neither reaches significance once `n_trials` is already in the model, `session_type` isn't explaining anything trial count doesn't already explain. The `n_trials` row's own coefficient/p-value shows how much of the metric trial count alone accounts for. The printed "Direct DCZ vs. saline contrast" t-test is the most relevant single number -- it's the saline-vs-dcz difference specifically, after adjusting for trial count.

- **Input:** `landmark_lookup`, `session_catalog`.
- **Output:** `calib_df`, `landmark_metric_test_results` (`{'fraction_preferring': model_result, 'frac_L4': model_result}`).

In [150]:
def get_session_trial_count(tseries_dir):
    """
    Read a session's trial count straight from its preproc.h5. Reimplemented
    unchanged from Phase 4's Function 4.5.
    """
    preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {tseries_dir}")
    with h5py.File(preproc_files[0], 'r') as f:
        n_trials = f['spatial_activity'].shape[1]
    return n_trials


def test_condition_effect_on_landmark_metric(calib_df, fraction_col, weight_col='n_cells',
                                             reference_level='baseline'):
    """
    Weighted least squares: fraction_col ~ n_trials + C(session_type),
    weighted by weight_col. See markdown above.

    Parameters
    ----------
    calib_df : pandas.DataFrame
        Needs fraction_col, n_trials, session_type, weight_col.
    fraction_col : str
    weight_col : str
    reference_level : str

    Returns
    -------
    model_result : statsmodels regression results object
    """
    df = calib_df.copy()
    other_levels = [c for c in df['session_type'].unique() if c != reference_level]
    df['session_type'] = pd.Categorical(df['session_type'], categories=[reference_level] + other_levels)

    formula = f"{fraction_col} ~ n_trials + C(session_type)"
    model_result = smf.wls(formula, data=df, weights=df[weight_col]).fit()

    print(f"\n=== {fraction_col} ~ n_trials + condition (weighted by {weight_col}, "
          f"reference='{reference_level}') ===")
    print(model_result.summary().tables[1])

    param_names = list(model_result.params.index)
    dcz_name = next((p for p in param_names if 'dcz' in p.lower()), None)
    saline_name = next((p for p in param_names if 'saline' in p.lower()), None)
    if dcz_name and saline_name:
        contrast = f"{dcz_name} - {saline_name}"
        print(f"\nDirect DCZ vs. saline contrast:")
        print(model_result.t_test(contrast))

    return model_result


def test_all_landmark_metrics(calib_df, reference_level='baseline'):
    """
    Runs test_condition_effect_on_landmark_metric for fraction_preferring
    (weighted by n_cells) and frac_L4 (weighted by n_preferring).

    Parameters
    ----------
    calib_df : pandas.DataFrame
    reference_level : str

    Returns
    -------
    results : dict
        {'fraction_preferring': model_result, 'frac_L4': model_result}.
    """
    results = {}
    results['fraction_preferring'] = test_condition_effect_on_landmark_metric(
        calib_df, 'fraction_preferring', weight_col='n_cells', reference_level=reference_level)
    results['frac_L4'] = test_condition_effect_on_landmark_metric(
        calib_df, 'frac_L4', weight_col='n_preferring', reference_level=reference_level)
    return results


# --- Build calib_df from all 15 discovered sessions ---
rows = []
for label, landmark_df in landmark_lookup.items():
    info = session_catalog[label]
    n_trials = get_session_trial_count(info['tseries_dir'])
    n_cells = len(landmark_df)
    preferring = landmark_df[landmark_df['has_landmark_preference']]
    n_preferring = len(preferring)
    frac_L4 = ((preferring['preferred_landmark_position'] == LANDMARK_POSITIONS[-1]).sum() / n_preferring
              if n_preferring > 0 else np.nan)
    rows.append({
        'session_label': label,
        'session_type': info['session_type'],
        'n_trials': n_trials,
        'n_cells': n_cells,
        'n_preferring': n_preferring,
        'fraction_preferring': n_preferring / n_cells if n_cells > 0 else np.nan,
        'frac_L4': frac_L4,
    })
calib_df = pd.DataFrame(rows).sort_values('n_trials').reset_index(drop=True)
print(calib_df.to_string(index=False))

# --- Run the regression check ---
landmark_metric_test_results = test_all_landmark_metrics(calib_df, reference_level='baseline')

                                                                    session_label session_type  n_trials  n_cells  n_preferring  fraction_preferring  frac_L4
                 260728_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_3_SALINE       saline        10      584           338             0.578767 0.071006
                 260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE       saline        20      578           254             0.439446 0.133858
                 260726_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_2_SALINE       saline        25      537           288             0.536313 0.093750
260801_JSY_JSY090_LongitudinalImaging_DREADD_StationaryOpenLoop_Saline_DCZ_SALINE       saline        26      576           316             0.548611 0.060127
    260730_JSY_JSY090_LongitudinalImaging_DREADD_ActiveOpenLoop_Saline_DCZ_SALINE       saline        28      626           343             0.547923 0.093294
                                                    

## Diagnostic — paired t-test: is the saline-vs-dcz drop reproducible (population + per-layer)?

The earlier population-level Wilcoxon signed-rank test (n=5 pairs, `overall_frac_L4`) capped at p=0.0625 -- Wilcoxon only uses each pair's *sign*, not its magnitude, so it can't distinguish "barely consistent" from "large and consistent." Given the drop is both large (mean ~6 percentage points) and low-variance across the 5 pairs, a **paired t-test** (which uses the actual values, not just ranks) is the appropriately powerful test here -- n=5 *pairs* is genuinely the correct independent unit (unlike the cell-level chi-square tests, which were pseudo-replicated and have already been removed from this notebook).

Result: paired t-test on the population-level metric reaches p<0.001, and running the same test separately per layer shows it holds in **every layer** (L2/3, L4, L5, L6), with L5 the strongest and most consistent and L6 consistently the weakest -- matching L6's recurring anomalous behavior in every other diagnostic in this phase (DCZ1's non-significant chi-square, Active_OL's near-zero drop).

**Read this correctly**: a significant paired t-test confirms the saline-vs-dcz pattern is real and reproducible across pairs -- it does NOT resolve whether it's attributable to DCZ specifically rather than trial count. Trial count differs systematically *within* every single pair (saline sessions are always the shorter ones), so pairing controls for between-pair differences (which day, which protocol) but cannot separate condition from trial count *within* a pair. That's what the WLS regression (with `n_trials` as an explicit covariate) already checked, and its direct DCZ-vs-saline contrast was not significant (p=0.279) -- these two results aren't a contradiction, they're answering different questions ("is this real" vs. "is this because of the drug").

- **Input:** `pop_df` (population-level), `landmark_composition_results` (per-layer).
- **Output:** `paired_population_df`, per-layer `paired_layer_summary_df`.

In [151]:
def test_paired_significance_population(pop_df, metric_col=None, landmark_positions=LANDMARK_POSITIONS):
    """
    Paired t-test (and Wilcoxon, for comparison) on the population-level
    metric across the 5 saline/dcz pairs. See markdown above.

    Parameters
    ----------
    pop_df : pandas.DataFrame
        From compute_population_landmark_fraction.
    metric_col : str, optional
        Defaults to 'overall_frac_L{n}' (the last landmark).
    landmark_positions : list of float

    Returns
    -------
    paired : pandas.DataFrame
        One row per group: saline, dcz, diff.
    t_stat, p_ttest : float
    """
    if metric_col is None:
        metric_col = f'overall_frac_L{len(landmark_positions)}'

    saline_vals = pop_df.loc[pop_df['condition'] == 'saline'].set_index('group')[metric_col]
    dcz_vals = pop_df.loc[pop_df['condition'] == 'dcz'].set_index('group')[metric_col]
    paired = pd.DataFrame({'saline': saline_vals, 'dcz': dcz_vals}).dropna()
    paired['diff'] = paired['saline'] - paired['dcz']

    t_stat, p_ttest = ttest_rel(paired['saline'], paired['dcz'])
    w_stat, p_wilcoxon = wilcoxon(paired['saline'], paired['dcz'])

    print(f"Paired comparison on '{metric_col}' (n={len(paired)} pairs):")
    print(paired.to_string())
    print(f"\nPaired t-test: t={t_stat:.3f}, df={len(paired) - 1}, p={p_ttest:.6f}")
    print(f"Wilcoxon signed-rank (for comparison): stat={w_stat:.3f}, p={p_wilcoxon:.4f}")
    print("\nNOTE: significant here means the pattern is reproducible across pairs, NOT that it's")
    print("attributable to DCZ specifically rather than trial count (see the WLS regression cell).")

    return paired, t_stat, p_ttest


def test_paired_significance_by_layer(landmark_composition_results, landmark_positions=LANDMARK_POSITIONS):
    """
    Same paired t-test, run separately per layer, using each group's
    composition_df (frac_L{n} x fraction_preferring = population-level
    per-layer metric). See markdown above.

    Parameters
    ----------
    landmark_composition_results : dict
        {group_name: run_landmark_composition_analysis_for_group(...) result}.
    landmark_positions : list of float

    Returns
    -------
    summary_df : pandas.DataFrame
        One row per layer: n_pairs, mean_diff, std_diff, n_same_direction,
        paired_t_stat, paired_t_p, wilcoxon_p.
    layer_paired_data : dict
        {layer: DataFrame(group, saline, dcz)}.
    """
    last_col = f'frac_L{len(landmark_positions)}'
    first_result = next(iter(landmark_composition_results.values()))
    layer_order = _layer_order(first_result['composition_df']['layer'].unique())

    rows = []
    layer_paired_data = {}
    for layer in layer_order:
        saline_vals, dcz_vals, group_names = [], [], []
        for group_name, result in landmark_composition_results.items():
            comp = result['composition_df']
            layer_rows = comp[comp['layer'] == layer].set_index('condition')
            if 'saline' in layer_rows.index and 'dcz' in layer_rows.index:
                s = layer_rows.loc['saline']
                d = layer_rows.loc['dcz']
                saline_vals.append(s['fraction_preferring'] * s[last_col])
                dcz_vals.append(d['fraction_preferring'] * d[last_col])
                group_names.append(group_name)

        saline_vals = np.array(saline_vals)
        dcz_vals = np.array(dcz_vals)
        diffs = saline_vals - dcz_vals

        t_stat, p_ttest = ttest_rel(saline_vals, dcz_vals)
        w_stat, p_wilcoxon = wilcoxon(saline_vals, dcz_vals)

        layer_paired_data[layer] = pd.DataFrame(
            {'group': group_names, 'saline': saline_vals, 'dcz': dcz_vals})

        rows.append({
            'layer': layer, 'n_pairs': len(saline_vals),
            'mean_diff': diffs.mean(), 'std_diff': diffs.std(ddof=1),
            'n_same_direction': int((diffs > 0).sum()),
            'paired_t_p': p_ttest, 'wilcoxon_p': p_wilcoxon,
        })

    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False))
    return summary_df, layer_paired_data

In [152]:
# --- Run the paired t-tests: population-level, then per-layer ---
paired_population_df, pop_t_stat, pop_p_ttest = test_paired_significance_population(pop_df)

print("\n" + "=" * 90 + "\n")

paired_layer_summary_df, layer_paired_data = test_paired_significance_by_layer(landmark_composition_results)

Paired comparison on 'overall_frac_L4' (n=5 pairs):
                 saline       dcz      diff
group                                      
Active_OL      0.051118  0.028846  0.022272
DCZ1           0.058824  0.090776 -0.031952
DCZ2           0.050279  0.056300 -0.006021
DCZ3           0.041096  0.109065 -0.067969
Stationary_OL  0.032986  0.016717  0.016269

Paired t-test: t=-0.811, df=4, p=0.462989
Wilcoxon signed-rank (for comparison): stat=5.000, p=0.6250

NOTE: significant here means the pattern is reproducible across pairs, NOT that it's
attributable to DCZ specifically rather than trial count (see the WLS regression cell).


layer  n_pairs  mean_diff  std_diff  n_same_direction  paired_t_p  wilcoxon_p
 L2/3        5  -0.011801  0.077554                 3    0.750784      1.0000
   L4        5   0.000575  0.007276                 2    0.868420      1.0000
   L5        5  -0.004252  0.035387                 2    0.801452      1.0000
   L6        5  -0.041063  0.050663              

## Cross-animal comparison — does JSY093's paired-t-test result replicate in JSY090?

JSY093: population-level paired t-test p=0.000855, every layer significant (p=0.0017–0.0200). JSY090: population-level p=0.463, no layer significant, and same-direction agreement close to 50/50 in every layer rather than JSY093's 5/5 — a genuine null, not an underpowered version of the same trend.

This loads both animals' *already-saved* paired t-test results (from `save_all_phase6_outputs`, run separately for each animal) rather than re-running the full pipeline twice in one cell — requires `save_all_phase6_outputs` to have already been run once with `TEST_ANIMAL_DIR` pointed at JSY093 and once at JSY090.

One thing to hold in mind reading this table: JSY093 is also the one animal where `frac_L4` showed a significant trial-count relationship in the WLS regression (p=0.012) — JSY090 showed none (p=0.619). So "JSY093 shows a significant paired effect, JSY090 doesn't" is consistent with a responder/non-responder story (matching Phase 4/5's characterization from the actual SMI effect), but it's equally consistent with "this metric happens to be more trial-count-sensitive in JSY093's specific set of recordings." The two explanations aren't distinguishable from this comparison alone.

- **Input:** `ANIMAL_DIRS_PHASE6` (animal directories).
- **Output:** `combined_pop_paired_df`, `combined_layer_paired_df`.

In [ ]:
def load_paired_ttest_results_for_animal(animal_dir, animal_id):
    """
    Loads one animal's already-saved paired t-test results
    (from save_all_phase6_outputs) rather than recomputing them, tagging
    each row with animal_id. See markdown above.

    Parameters
    ----------
    animal_dir : str
    animal_id : str

    Returns
    -------
    pop_df : pandas.DataFrame
        Population-level paired comparison (saline, dcz, diff), one row
        per group.
    layer_df : pandas.DataFrame
        Per-layer paired comparison summary.
    """
    output_dir = os.path.join(animal_dir, 'Phase6_LandmarkPreference_Results')

    pop_df = pd.read_csv(os.path.join(output_dir, 'paired_ttest_population.csv'), index_col=0)
    pop_df['animal_id'] = animal_id

    layer_df = pd.read_csv(os.path.join(output_dir, 'paired_ttest_by_layer.csv'))
    layer_df['animal_id'] = animal_id

    return pop_df, layer_df


def compare_paired_ttest_across_animals(animal_dirs):
    """
    Loads and combines every animal's saved paired t-test results
    (population + per-layer) into side-by-side comparison tables. See
    markdown above.

    Parameters
    ----------
    animal_dirs : dict
        {animal_id: animal_dir}.

    Returns
    -------
    combined_pop_df : pandas.DataFrame
    combined_layer_df : pandas.DataFrame
    """
    pop_parts, layer_parts = [], []
    for animal_id, animal_dir in animal_dirs.items():
        pop_df, layer_df = load_paired_ttest_results_for_animal(animal_dir, animal_id)
        pop_parts.append(pop_df)
        layer_parts.append(layer_df)

    combined_pop_df = pd.concat(pop_parts)
    combined_layer_df = pd.concat(layer_parts, ignore_index=True)

    print("Population-level paired t-test, by animal:")
    print(combined_pop_df.to_string())
    print("\nPer-layer paired t-test, by animal:")
    print(combined_layer_df.to_string(index=False))

    return combined_pop_df, combined_layer_df

In [ ]:
# --- Compare both animals (requires save_all_phase6_outputs already run for each) ---
ANIMAL_DIRS_PHASE6 = {
    'JSY093': r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD",
    'JSY090': r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD",
}

combined_pop_paired_df, combined_layer_paired_df = compare_paired_ttest_across_animals(ANIMAL_DIRS_PHASE6)

## Save-outputs — `save_all_phase6_outputs`

Same reasoning as every earlier phase's save step: nothing generated in this notebook has been persisted to disk yet — the composition tables, chi-square results, all the figures, the open-loop/closed-loop summary, the paired t-test results, Option B's results, and the trial-count regression models only ever lived in the kernel's memory.

Built as small, reusable pieces (reimplemented from Phase 4/5's Function 4.8/5.10, since digit-prefixed module filenames can't be imported):
1. **`save_dataframe_csv`** / **`save_figure_png`** / **`_json_safe`** / **`save_json`** — generic helpers, creating `output_dir` if needed.
2. **`save_all_phase6_outputs`** — saves everything in one call. Figures are regenerated fresh from their underlying data rather than reused from whatever's currently in kernel memory, since several plots in this notebook share a generic `fig` variable name that gets overwritten by later cells — regenerating avoids saving a stale/wrong figure.

Saved under `{ANIMAL_DIR}/Phase6_LandmarkPreference_Results/`:
- `{group}_landmark_composition_by_layer.csv` / `.json` (chi2) / `_plot.png` — per comparison group.
- `population_landmark_fractions.csv`, `landmark_last_slope_population.png`, `landmark_last_slope_by_layer.png`.
- `openloop_vs_closedloop_effect_summary.csv`.
- `paired_ttest_population.csv`, `paired_ttest_by_layer.csv`.
- `option_b_smi_landmark1_vs_last.csv`, `option_b_smi_landmark1_vs_last_plot.png`.
- `trial_count_calibration.csv`, `wls_regression_{metric}.txt` (full regression summaries).

- **Input:** `output_dir`, plus every result object already computed above (`landmark_composition_results`, `pop_df`, `openloop_effect_summary`, `paired_population_df`, `paired_layer_summary_df`, `landmark_smi_summary_df`, `calib_df`, `landmark_metric_test_results`).
- **Output:** `saved_paths` (dict of every path written).

In [153]:
def save_dataframe_csv(df, output_dir, filename, index=False):
    """
    Save a DataFrame to {output_dir}/{filename}, creating output_dir if
    needed. index=True for tables whose index is meaningful.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    df.to_csv(save_path, index=index)
    print(f"Saved -> {save_path}")
    return save_path


def save_figure_png(fig, output_dir, filename, dpi=150):
    """
    Save a matplotlib figure to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"Saved -> {save_path}")
    return save_path


def _json_safe(obj):
    """Recursively convert numpy scalar types to native Python for json.dump."""
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return _json_safe(obj.tolist())
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def save_json(data, output_dir, filename):
    """
    Save a JSON-serializable dict to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    with open(save_path, 'w') as f:
        json.dump(_json_safe(data), f, indent=2)
    print(f"Saved -> {save_path}")
    return save_path


def save_all_phase6_outputs(output_dir, landmark_composition_results, pop_df, openloop_effect_summary,
                            paired_population_df, paired_layer_summary_df, landmark_smi_summary_df,
                            calib_df, landmark_metric_test_results):
    """
    Saves everything Phase 6 generated for one animal. See markdown above.

    Parameters
    ----------
    output_dir : str
        e.g. os.path.join(ANIMAL_DIR, 'Phase6_LandmarkPreference_Results').
    landmark_composition_results : dict
        {group_name: {'composition_df', 'chi2_results', ...}}.
    pop_df : pandas.DataFrame
    openloop_effect_summary : pandas.DataFrame
    paired_population_df : pandas.DataFrame
    paired_layer_summary_df : pandas.DataFrame
    landmark_smi_summary_df : pandas.DataFrame
    calib_df : pandas.DataFrame
    landmark_metric_test_results : dict
        {'fraction_preferring': model_result, 'frac_L4': model_result}.

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {}

    # 1. Per-group composition results
    for group_name, result in landmark_composition_results.items():
        saved_paths[f'{group_name}_composition'] = save_dataframe_csv(
            result['composition_df'], output_dir, f'{group_name}_landmark_composition_by_layer.csv')

        chi2_summary = {layer: ({'chi2': r['chi2'], 'p': r['p'], 'dof': r['dof'], 'conditions': r['conditions']}
                                if r is not None else None)
                        for layer, r in result['chi2_results'].items()}
        saved_paths[f'{group_name}_chi2'] = save_json(
            chi2_summary, output_dir, f'{group_name}_landmark_composition_chi2.json')

        fig = plot_landmark_composition_by_layer(
            result['composition_df'], title=f'{group_name} -- landmark composition by layer')
        saved_paths[f'{group_name}_composition_plot'] = save_figure_png(
            fig, output_dir, f'{group_name}_landmark_composition_by_layer_plot.png')
        plt.close(fig)

    # 2. Population-level summary + figures
    saved_paths['pop_df'] = save_dataframe_csv(pop_df, output_dir, 'population_landmark_fractions.csv')

    fig1 = plot_landmark_last_slope_population(pop_df)
    saved_paths['slope_population_fig'] = save_figure_png(
        fig1, output_dir, 'landmark_last_slope_population.png')
    plt.close(fig1)

    fig2 = plot_landmark_last_slope_by_layer(landmark_composition_results)
    saved_paths['slope_layer_fig'] = save_figure_png(
        fig2, output_dir, 'landmark_last_slope_by_layer.png')
    plt.close(fig2)

    # 3. Open-loop vs closed-loop
    saved_paths['openloop_effect'] = save_dataframe_csv(
        openloop_effect_summary, output_dir, 'openloop_vs_closedloop_effect_summary.csv')

    # 4. Paired t-tests
    saved_paths['paired_population'] = save_dataframe_csv(
        paired_population_df, output_dir, 'paired_ttest_population.csv', index=True)
    saved_paths['paired_layer'] = save_dataframe_csv(
        paired_layer_summary_df, output_dir, 'paired_ttest_by_layer.csv')

    # 5. Option B
    saved_paths['option_b_summary'] = save_dataframe_csv(
        landmark_smi_summary_df, output_dir, 'option_b_smi_landmark1_vs_last.csv')
    smi_fig = plot_smi_landmark1_vs_last(landmark_smi_summary_df)
    saved_paths['option_b_fig'] = save_figure_png(
        smi_fig, output_dir, 'option_b_smi_landmark1_vs_last_plot.png')
    plt.close(smi_fig)

    # 6. Trial-count regression
    saved_paths['calib_df'] = save_dataframe_csv(calib_df, output_dir, 'trial_count_calibration.csv')

    os.makedirs(output_dir, exist_ok=True)
    for metric, model_result in landmark_metric_test_results.items():
        txt_path = os.path.join(output_dir, f'wls_regression_{metric}.txt')
        with open(txt_path, 'w') as f:
            f.write(str(model_result.summary()))
        print(f"Saved -> {txt_path}")
        saved_paths[f'wls_{metric}'] = txt_path

    print(f"\nSaved {len(saved_paths)} Phase 6 output(s) to {output_dir}")
    return saved_paths

In [154]:
# --- Save everything Phase 6 generated for this animal ---
PHASE6_OUTPUT_DIR = os.path.join(TEST_ANIMAL_DIR, 'Phase6_LandmarkPreference_Results')

saved_paths = save_all_phase6_outputs(
    PHASE6_OUTPUT_DIR, landmark_composition_results, pop_df, openloop_effect_summary,
    paired_population_df, paired_layer_summary_df, landmark_smi_summary_df,
    calib_df, landmark_metric_test_results)

Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase6_LandmarkPreference_Results\Active_OL_landmark_composition_by_layer.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase6_LandmarkPreference_Results\Active_OL_landmark_composition_chi2.json
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase6_LandmarkPreference_Results\Active_OL_landmark_composition_by_layer_plot.png
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase6_LandmarkPreference_Results\DCZ1_landmark_composition_by_layer.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase6_LandmarkPreference_Results\DCZ1_landmark_composition_chi2.json
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase6_LandmarkPreference_Results\DCZ1_landmark_composition_by_layer_plot.png
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase6_LandmarkPreference